# EDA for Covariates

In [ ]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", None)

# PUT GITHUB ROOT PATH HERE
root_dir = "/Users/albert/Library/CloudStorage/OneDrive-andrew.cmu.edu/Documents/CMU 26-27/Undergraduate Research/Victimization/Code/capstone_victimization"
dir = root_dir + "/Data and Resources"
os.chdir(dir)
data = pd.read_csv(os.path.join(dir, "final_data.csv"))
print(data.shape)
print(data.columns)

(8984, 28)
Index(['respondent_id', 'timely_college', 'violent_victimization_18_24',
       'earliest_arrest_date', 'total_number_of_incarcerations',
       'r_threatened_to_be_hurt_at_school', 'r_in_a_fight_at_school',
       'days_r_absent_from_school', 'teachers_good',
       'discipline_is_fair_agree_disagree', 'percent_of_peers_who_smoke',
       'percent_of_peers_who_get_drunk_1_times_a_month',
       'percent_of_peers_belong_to_a_gang',
       'percent_of_peers_who_have_had_sex', 'what_year_first_arrest_arrest_10',
       'bullying_victim_before_age_12', 'r_victim_of_bullying_12_18_years',
       'percent_chance_r_victim_of_violent_crime_by_next_year', 'sex',
       'r_ever_live_through_hard_times', 'household_net_worth',
       'household_size', 'biological_father_education',
       'biological_mother_education', 'race_ethnicity',
       'asvab_math_verbal_score_percent', 'mother_serve_prison_sentence',
       'father_serve_prison_sentence'],
      dtype='str')


In [2]:
covariates = [
    col for col in data.columns
    if col not in ["respondent_id", "timely_college", "violent_victimization_18_24"]
]

### Missingness

In [3]:
data[covariates].isna().sum()

earliest_arrest_date                                        0
total_number_of_incarcerations                              0
r_threatened_to_be_hurt_at_school                           0
r_in_a_fight_at_school                                      0
days_r_absent_from_school                                   0
teachers_good                                               0
discipline_is_fair_agree_disagree                           0
percent_of_peers_who_smoke                                  0
percent_of_peers_who_get_drunk_1_times_a_month              0
percent_of_peers_belong_to_a_gang                           0
percent_of_peers_who_have_had_sex                           0
what_year_first_arrest_arrest_10                            0
bullying_victim_before_age_12                               0
r_victim_of_bullying_12_18_years                         2025
percent_chance_r_victim_of_violent_crime_by_next_year       0
sex                                                         0
r_ever_l

### Descriptive Statistics

In [9]:
df_desc = data[covariates].copy()

# Replace negative values with NaN only for numeric columns
numeric_cols = df_desc.select_dtypes(include="number").columns

df_desc[numeric_cols] = df_desc[numeric_cols].mask(
    df_desc[numeric_cols] < 0
)

# Convert numeric columns to nullable integers
df_desc[numeric_cols] = df_desc[numeric_cols].astype("Int64")

# Descriptive statistics
desc = df_desc.describe(include="all").T

# Add missing count
desc["missing"] = df_desc.isna().sum()

desc

,count,unique,top,freq,mean,std,min,25%,50%,75%,max,missing
earliest_arrest_date,3215.0,<NA>,<NA>,<NA>,200124.729082,562.162503,198703.0,199709.0,200007.0,200407.0,202404.0,5769
total_number_of_incarcerations,8948.0,<NA>,<NA>,<NA>,0.210326,0.758556,0.0,0.0,0.0,0.0,11.0,36
r_threatened_to_be_hurt_at_school,8869.0,<NA>,<NA>,<NA>,0.97745,5.226506,0.0,0.0,0.0,0.0,99.0,115
r_in_a_fight_at_school,8872.0,<NA>,<NA>,<NA>,0.401262,1.707949,0.0,0.0,0.0,0.0,50.0,112
days_r_absent_from_school,8687.0,<NA>,<NA>,<NA>,4.889375,7.588716,0.0,1.0,3.0,6.0,200.0,297
teachers_good,8965.0,<NA>,<NA>,<NA>,1.984718,0.622975,1.0,2.0,2.0,2.0,4.0,19
discipline_is_fair_agree_disagree,8943.0,<NA>,<NA>,<NA>,2.186291,0.782814,1.0,2.0,2.0,3.0,4.0,41
percent_of_peers_who_smoke,8871.0,<NA>,<NA>,<NA>,2.611656,1.282488,1.0,1.0,3.0,4.0,5.0,113
percent_of_peers_who_get_drunk_1_times_a_month,8799.0,<NA>,<NA>,<NA>,2.136152,1.246358,1.0,1.0,2.0,3.0,5.0,185
percent_of_peers_belong_to_a_gang,8812.0,<NA>,<NA>,<NA>,1.594757,0.980798,1.0,1.0,1.0,2.0,5.0,172


In [5]:
for col in covariates:

    # Categorical / binary variables
    if data[col].nunique(dropna=False) <= 10:

        counts = (
            data[col]
            .fillna("Missing")
            .value_counts()
            .reset_index()
        )
        counts.columns = [col, "Count"]

        fig = px.bar(
            counts,
            x=col,
            y="Count",
            title=f"Distribution of {col}"
        )

    # Continuous variables
    else:

        fig = px.histogram(
            data,
            x=col,
            nbins=20,
            title=f"Distribution of {col}"
        )

    fig.update_layout(
        width=700,
        height=400
    )

    fig.show()

In [ ]:
outcome_col = "violent_victimization_18_24"
has_outcome = data[outcome_col].notna()

for col in covariates:

    # Categorical / binary variables: victimization RATE by category
    if data[col].nunique(dropna=False) <= 10:

        sub = data.loc[has_outcome, [col, outcome_col]].copy()
        sub[col] = sub[col].fillna("Missing")

        summary = sub.groupby(col)[outcome_col].agg(n="count", n_victimized="sum")
        summary["rate_pct"] = (summary["n_victimized"] / summary["n"] * 100).round(2)
        summary = summary.sort_index()

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.bar(range(len(summary)), summary["rate_pct"])
        for i, (rate, n, nv) in enumerate(
            zip(summary["rate_pct"], summary["n"], summary["n_victimized"])
        ):
            ax.text(i, rate, f"{rate:.1f}%\n({int(nv):,}/{int(n):,})",
                    ha="center", va="bottom", fontsize=8)
        ax.set_xticks(range(len(summary)))
        ax.set_xticklabels(summary.index, rotation=30, ha="right", fontsize=8)
        ax.set_ylabel("% victimized (ages 18-24)")
        ax.set_title(f"Victimization rate by {col}\n(n = {len(sub):,} with a defined outcome)")
        ax.margins(y=0.18)

    # Continuous variables: distribution split by victimization status
    else:
        fig, ax = plt.subplots(figsize=(7, 4))
        for label, flag in [("Not victimized", 0), ("Victimized", 1)]:
            s = data.loc[has_outcome & data[outcome_col].eq(flag), col].dropna()
            if len(s):
                ax.hist(s, bins=20, alpha=0.5, density=True, label=f"{label} (n={len(s):,})")
        ax.set_xlabel(col)
        ax.set_ylabel("Density")
        ax.set_title(f"Distribution of {col} by victimization status")
        ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()